# ICT-15j — Discriminant Čech par nerf simplicial (GUDHI)

## Cadrage

### Question et hypothèse

Le verdict de `ict.cech_obstruction.verdict` était dominé par `s2_over_s1` et `effective_rank`. Le rang spectral suffit-il à caractériser l’obstruction, ou le nerf simplicial peut-il distinguer un relief topologique que la SVD manque ? Sur le contre-exemple Axelrod, le cocycle et `obstruction_ratio` valent zéro, alors que le verdict SVD reste `NON_TRIVIAL` (rang 2) : la SVD consulte le rang spectral, pas la cohomologie du nerf.

Le test construit le nerf simplicial sur 30 fenêtres et trois proxys (`spectral_gap`, `sensitivity_mean`, `sensitivity_max`) par substrat, puis compte les classes H¹ persistantes (`b1`). Il examine si `b1 ≥ 1` sur au moins un substrat et si `b1` diverge du verdict SVD.

## Protocole falsifiable

- discriminer les quatre substrats (`gray_scott`, `axelrod`, `grokking`, `may`), avec un verdict non trivial sur l’Axelrod à cocycle nul;
- diverger de `s2_over_s1` sur au moins un substrat, pour écarter un proxy redondant;
- pré-enregistrer les verdicts `TRIVIAL`, `NON_TRIVIAL` ou `PROXY_REDUNDANT` avant la mesure;
- borner l’effort à ≤ 30 minutes ICT-grade, sans GPU.

## Observable et limite connue

Une version initiale concluait « b1(axl)=0 falsifie la SVD ». Cette observation provenait d’un observable dégénéré : la dynamique réplicateur converge monotoniquement, l’argmax ne change plus, et la série binaire « stabilité du dominant » devient quasi constante. Le nerf sature (4 060 triangles) et `b1=0` par construction, indépendamment du système sous-jacent. Le substrat Axelrod est donc régénéré avec une population évolutive (sélection-mutation Wright–Fisher, voir ICT-13 §6) et un observable de taux de coopération quantifié sur huit symboles. L’ancien substrat reste la baseline dégénérée dans le banc; la comparaison côte à côte rend cette limite visible.

La mesure reste descriptive : « nombre de classes H¹ persistantes dans le nerf simplicial sur les sections locales des proxys ». Le terme « discriminant Čech » nomme l’outil, sans attribuer de nom phénoménologique à l’objet.

> **Statut épistémique** — **Sans verdict à ce jour** : aucune ligne de la [matrice de dissociations](../../../../docs/ict/dissociations-matrix.md) ne concerne ce notebook ; son statut épistémique sera porté par la matrice le cas échéant.

In [1]:
// Imports et substrats pilotes
using System;
using System.Linq;
using System.Collections.Generic;
using System.Globalization;
CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.CurrentUICulture = CultureInfo.InvariantCulture;

// Générateur reproductible : les tirages diffèrent de NumPy.
public class Rng15j
{
    ulong state;
    public Rng15j(ulong seed) { state = seed == 0 ? 1UL : seed; }
    public ulong Next64() { state ^= state >> 12; state ^= state << 25; state ^= state >> 27; return state * 2685821657736338717UL; }
    public double Uniform() { return (Next64() >> 11) * (1.0 / 9007199254740992.0); }
    public int Integer(int n) { return (int)(Uniform() * n); }
    public double Normal() { double u = Math.Max(Uniform(), 1e-300); return Math.Sqrt(-2.0 * Math.Log(u)) * Math.Cos(2.0 * Math.PI * Uniform()); }
    public double Gamma(double shape) {
        if (shape < 1.0) return Gamma(shape + 1.0) * Math.Pow(Uniform(), 1.0 / shape);
        double d = shape - 1.0 / 3.0, c = 1.0 / Math.Sqrt(9.0 * d);
        while (true) { double x = Normal(), v = 1.0 + c * x; if (v <= 0) continue; v = v * v * v; double u = Uniform(); if (u < 1.0 - 0.0331 * x * x * x * x || Math.Log(Math.Max(u, 1e-300)) < 0.5 * x * x + d * (1.0 - v + Math.Log(v))) return d * v; }
    }
    public void Shuffle(int[] a) { for (int i = a.Length - 1; i > 0; i--) { int j = Integer(i + 1); int t = a[i]; a[i] = a[j]; a[j] = t; } }
}

public static class Ict15j
{
    public static double Quantile(double[] values, double q) {
        var a = values.OrderBy(x => x).ToArray(); double p = (a.Length - 1) * q; int i = (int)Math.Floor(p); int j = Math.Min(i + 1, a.Length - 1); return a[i] + (p - i) * (a[j] - a[i]);
    }
    public static int[] Digitize(double[] values, int bins) {
        var cuts = Enumerable.Range(1, bins - 1).Select(i => Quantile(values, (double)i / bins)).ToArray();
        return values.Select(x => cuts.Count(c => x >= c)).ToArray();
    }
    public static int Move(int strategy, List<int> own, List<int> opp, Rng15j rng) {
        if (strategy == 0) return 0;
        if (strategy == 1) return 1;
        if (strategy == 2) return opp.Count == 0 ? 0 : opp[opp.Count - 1];
        if (strategy == 3) return opp.Count == 0 || opp[opp.Count - 1] == 0 || rng.Uniform() < 1.0 / 3.0 ? 0 : 1;
        if (strategy == 4) { if (own.Count == 0) return 0; int a = own[own.Count - 1], b = opp[opp.Count - 1]; double gain = Payoff(a, b); return gain >= 3.0 ? a : 1 - a; }
        return opp.Contains(1) ? 1 : 0;
    }
    public static double Payoff(int a, int b) { return a == 0 ? (b == 0 ? 3.0 : 0.0) : (b == 0 ? 5.0 : 1.0); }
    public static (double, double, int) Match(int s1, int s2, int rounds, double noise, Rng15j rng, Rng15j strategyRng) {
        var a = new List<int>(); var b = new List<int>(); double g1 = 0, g2 = 0; int coop = 0;
        for (int t = 0; t < rounds; t++) { int x = Move(s1, a, b, strategyRng), y = Move(s2, b, a, strategyRng); if (noise > 0) { if (rng.Uniform() < noise) x = 1 - x; if (rng.Uniform() < noise) y = 1 - y; } g1 += Payoff(x, y); g2 += Payoff(y, x); if (x == 0) coop++; if (y == 0) coop++; a.Add(x); b.Add(y); }
        return (g1 / rounds, g2 / rounds, coop);
    }
    public static double[,] PayoffMatrix(Rng15j rng) {
        var a = new double[6, 6]; for (int i = 0; i < 6; i++) for (int j = 0; j < 6; j++) for (int r = 0; r < 3; r++) a[i, j] += Match(i, j, 200, 0.0, rng, rng).Item1 / 3.0; return a;
    }
    public static int[] Replicator(double[,] a, Rng15j rng) {
        double[] x = Enumerable.Range(0, 6).Select(_ => rng.Gamma(50.0)).ToArray(); double sum = x.Sum(); for (int i = 0; i < 6; i++) x[i] /= sum;
        int previous = Array.IndexOf(x, x.Max()); var result = new int[400];
        for (int t = 0; t < 400; t++) { var f = new double[6]; for (int i = 0; i < 6; i++) for (int j = 0; j < 6; j++) f[i] += a[i, j] * x[j]; double avg = Enumerable.Range(0, 6).Sum(i => f[i] * x[i]); if (avg >= 1e-12) { for (int i = 0; i < 6; i++) x[i] = Math.Max(0.0, x[i] * f[i] / avg); sum = x.Sum(); if (sum > 1e-12) for (int i = 0; i < 6; i++) x[i] /= sum; } int next = Array.IndexOf(x, x.Max()); result[t] = next == previous ? 1 : 0; previous = next; }
        return result;
    }
    public static double[] Evolve(Rng15j rng) {
        int n = 60; var pop = Enumerable.Range(0, n).Select(_ => rng.Integer(6)).ToArray(); var coop = new double[401];
        for (int gen = 0; gen <= 400; gen++) { var perm = Enumerable.Range(0, n).ToArray(); rng.Shuffle(perm); var scores = new double[n]; int total = 0;
            for (int k = 0; k < n; k += 2) { int i = perm[k], j = perm[k + 1]; var m = Match(pop[i], pop[j], 30, 0.02, rng, rng); scores[i] = m.Item1; scores[j] = m.Item2; total += m.Item3; }
            coop[gen] = (double)total / (n * 30); if (gen == 400) break;
            var weights = scores.Select(x => Math.Max(x, 1e-6)).ToArray(); double sum = weights.Sum(); var children = new int[n];
            for (int i = 0; i < n; i++) { double r = rng.Uniform() * sum, c = 0; int parent = n - 1; for (int j = 0; j < n; j++) { c += weights[j]; if (r < c) { parent = j; break; } } children[i] = pop[parent]; }
            var mutate = new bool[n]; for (int i = 0; i < n; i++) mutate[i] = rng.Uniform() < 0.05; for (int i = 0; i < n; i++) if (mutate[i]) children[i] = rng.Integer(6); pop = children;
        }
        return coop;
    }
    public static int[] GrayScott(Rng15j rng) {
        int n = 64; var u = new double[n, n]; var v = new double[n, n];
        for (int i = 0; i < n; i++) for (int j = 0; j < n; j++) u[i, j] = Math.Clamp((i >= 26 && i < 38 && j >= 26 && j < 38 ? 0.5 : 1.0) + 0.02 * rng.Normal(), 0, 1);
        for (int i = 0; i < n; i++) for (int j = 0; j < n; j++) v[i, j] = Math.Clamp((i >= 26 && i < 38 && j >= 26 && j < 38 ? 0.25 : 0.0) + 0.02 * rng.Normal(), 0, 1);
        for (int t = 0; t < 800; t++) { var un = new double[n, n]; var vn = new double[n, n]; for (int i = 0; i < n; i++) for (int j = 0; j < n; j++) { int ip = (i + 1) % n, im = (i + n - 1) % n, jp = (j + 1) % n, jm = (j + n - 1) % n; double lu = u[im,j] + u[ip,j] + u[i,jm] + u[i,jp] - 4 * u[i,j], lv = v[im,j] + v[ip,j] + v[i,jm] + v[i,jp] - 4 * v[i,j], reaction = u[i,j] * v[i,j] * v[i,j]; un[i,j] = Math.Clamp(u[i,j] + 0.16 * lu - reaction + 0.035 * (1 - u[i,j]), 0, 1); vn[i,j] = Math.Clamp(v[i,j] + 0.08 * lv + reaction - 0.1 * v[i,j], 0, 1); } u = un; v = vn; }
        var states = new int[n*n]; for (int i = 0; i < n; i++) for (int j = 0; j < n; j++) states[i*n+j] = v[i,j] > 0.05 ? 1 : 0; return states;
    }
    public static double[] May(Rng15j rng) { var xs = new double[2000]; double x = 8; for (int t = 0; t < xs.Length; t++) { double rate = x * (1 - x / 10.0) - 1.5 * x * x / (x * x + 1.0); x = Math.Max(0, x + 0.01 * rate + 0.005 * rng.Normal()); xs[t] = x; } return xs; }

    // Wrappers proxy_signature (memes recettes que ICT-15c)
    public static double SpectralGap(int[] states, int k) {
        var p = new double[k,k]; for (int i = 0; i < k; i++) for (int j = 0; j < k; j++) p[i,j] = 1e-9;
        for (int t = 0; t + 1 < states.Length; t++) if (states[t] >= 0 && states[t] < k && states[t+1] >= 0 && states[t+1] < k) p[states[t], states[t+1]]++;
        for (int i = 0; i < k; i++) { double sum = 0; for (int j = 0; j < k; j++) sum += p[i,j]; for (int j = 0; j < k; j++) p[i,j] /= sum; }
        var l = new double[k,k]; for (int i = 0; i < k; i++) for (int j = 0; j < k; j++) if (i != j) { double w = (p[i,j] + p[j,i]) / 2; l[i,j] = -w; l[i,i] += w; }
        for (int iter = 0; iter < 10000; iter++) { int a = 0, b = 1; double max = 0; for (int i = 0; i < k; i++) for (int j = i + 1; j < k; j++) if (Math.Abs(l[i,j]) > max) { max = Math.Abs(l[i,j]); a = i; b = j; } if (max < 1e-14) break; double phi = 0.5 * Math.Atan2(2 * l[a,b], l[b,b] - l[a,a]), c = Math.Cos(phi), s = Math.Sin(phi); double aa = l[a,a], bb = l[b,b], ab = l[a,b]; for (int j = 0; j < k; j++) if (j != a && j != b) { double aj = l[a,j], bj = l[b,j]; l[a,j] = l[j,a] = c * aj - s * bj; l[b,j] = l[j,b] = s * aj + c * bj; } l[a,a] = c*c*aa - 2*s*c*ab + s*s*bb; l[b,b] = s*s*aa + 2*s*c*ab + c*c*bb; l[a,b] = l[b,a] = 0; }
        var eig = Enumerable.Range(0,k).Select(i => l[i,i]).OrderBy(x => x).ToArray(); return eig[1] - eig[0];
    }
    public static (double mean, double max) Sensitivity(int[] states, int k) {
        var ids = new Dictionary<int,int>(); var enc = new int[states.Length]; for (int i = 0; i < states.Length; i++) { if (!ids.ContainsKey(states[i])) { if (ids.Count >= k) throw new ArgumentException("Trop de symboles"); ids[states[i]] = ids.Count; } enc[i] = ids[states[i]]; }
        var adj = new bool[k,k]; for (int i = 0; i + 1 < enc.Length; i++) if (enc[i] != enc[i+1]) adj[enc[i],enc[i+1]] = adj[enc[i+1],enc[i]] = true;
        var degrees = new double[ids.Count]; for (int i = 0; i < ids.Count; i++) for (int j = 0; j < k; j++) if (adj[i,j]) degrees[i]++; return degrees.Length == 0 ? (0,0) : (degrees.Average(), degrees.Max());
    }
    public static Dictionary<string,List<double>> WindowedProxySignature(int[] states, int k, int nWindows = 30) {
        var result = new Dictionary<string,List<double>> { ["spectral_gap"] = new List<double>(), ["sensitivity_mean"] = new List<double>(), ["sensitivity_max"] = new List<double>() };
        for (int w = 0; w < nWindows; w++) { int start = w * states.Length / nWindows, end = Math.Max(start + 2, (w + 1) * states.Length / nWindows); var chunk = states.Skip(start).Take(Math.Max(0, end - start)).ToArray(); if (chunk.Length < 2) chunk = states.Skip(start).Take(4).ToArray();
            try { result["spectral_gap"].Add(SpectralGap(chunk,k)); } catch { result["spectral_gap"].Add(double.NaN); }
            try { var s = Sensitivity(chunk,k); result["sensitivity_mean"].Add(s.mean); result["sensitivity_max"].Add(s.max); } catch { result["sensitivity_mean"].Add(double.NaN); result["sensitivity_max"].Add(double.NaN); }
        }
        foreach (var key in result.Keys.ToArray()) { result[key] = result[key].Where(double.IsFinite).ToList(); while (result[key].Count < nWindows) result[key].Add(0); } return result;
    }
}
Console.WriteLine("nerve_discriminant loaded. Substrats pilotes : Gray-Scott, Axelrod, Grokking, May.");

nerve_discriminant loaded. Substrats pilotes : Gray-Scott, Axelrod, Grokking, May.


### Le nerf simplicial : b1 comme discriminant Čech

On construit, pour chaque substrat, un **nuage de 30 points** dans
R³ (un point par fenêtre × 3 proxys). Le **nerf simplicial Čech** (ici
l'approximation Rips, gudhi) sur ce nuage retient les simplexes dont les
arêtes sont de longueur ≤ ε ; **b1** est le nombre de **cycles 1-dim**
persistants dans la filtration.

Pour un nuage de 30 points quasi-regroupés en un seul cluster, le complexe
Rips est **contractile** et b1 = 0. Si les points dessinent une « boucle »
(anneau, lacet tordu), b1 ≥ 1.

**Mesure recommandée** : `b1_max_persistence` (la persistance maximale des
classes H^1 sur toute la filtration). Elle est **stable au choix du seuil**
contrairement au `b1` instantané à ε fixe.

**Sur l'axelrod, deux observables** (#12673) :

- **baseline dégénérée** (`axelrod_repl`) : la dynamique réplicateur converge
  de façon monotone, l'argmax ne change plus → série binaire quasi constante →
  nuage clusterisé → Rips contractile → b1 = 0 attendu **par construction** ;
- **substrat régénéré** (`axelrod`) : une population qui **évolue**
  (sélection-mutation, ICT-13 §6), observable = taux de coopération quantifié
  sur 8 symboles. La trajectoire oscille (équilibre polymorphe) → les fenêtres
  capturent des régimes distincts → on attend du relief, i.e. b1 > 0.

La comparaison des deux mesure ce que le discriminant détecte réellement :
la **forme de l'observable**, pas la richesse du système dessous.

In [2]:
// --- Substrat 1 : Gray-Scott (motif binarise, alphabet=2) ---
var seedRng = new Rng15j(20260720);
var gray_scott_states = Ict15j.GrayScott(seedRng);
Console.WriteLine($"Gray-Scott : {gray_scott_states.Length} pixels, sum={gray_scott_states.Sum()}");

// --- Substrat 2a : Axelrod baseline DEGENEREE (replicateur, #12257 historique) ---
// Consommee telle quelle : l'observable binaire "l'argmax a-t-il change" est
// quasi constant apres convergence -> nerf sature -> b1=0 par construction.
var rng = new Rng15j(20260720);
var A = Ict15j.PayoffMatrix(rng);
// x0 hors du point uniforme exact (point fixe du replicateur : la trajectoire
// y reste a l'epsilon machine pres et l'argmax ne departage que du bruit
// d'arrondi BLAS, non reproductible inter-machines -- #12318, meme geste que
// le correctif ICT-15j). Dirichlet(50) : proche de l'uniforme sans egalite
// exacte ; la baseline reste degeneree (argmax fige apres convergence).
var axelrod_repl_states = Ict15j.Replicator(A, rng);
Console.WriteLine($"Axelrod (repl baseline) : {axelrod_repl_states.Length} pas, sum={axelrod_repl_states.Sum()}");

// --- Substrat 2b : Axelrod REGENERE (population qui evolue, #12673) ---
// Wright-Fisher selection-mutation (ICT-13 section 6) : 60 agents, 400
// generations, mutation 5 %. Observable = taux de cooperation quantifie en
// 8 symboles (quantiles) -> la trajectoire a du relief.
var rngEvo = new Rng15j(20260720);
var coop = Ict15j.Evolve(rngEvo);
var axelrod_states = Ict15j.Digitize(coop, 8);
Console.WriteLine($"Axelrod (evo regenere)  : {axelrod_states.Length} pas, coop[{coop.Min():F3},{coop.Max():F3}], n_unique={axelrod_states.Distinct().Count()}");

// --- Substrat 3 : Grokking (marche biaisee crossover) ---
var grokking_states = new int[400];
for (int t = 0; t < grokking_states.Length; t++) grokking_states[t] = t < 200 ? rng.Integer(4) : (rng.Uniform() < 0.90 ? 0 : 1 + rng.Integer(3));
Console.WriteLine($"Grokking : {grokking_states.Length} pas, n_unique={grokking_states.Distinct().Count()}");

// --- Substrat 4 : May (SDE bistable) ---
var xsMay = Ict15j.May(new Rng15j(20260720));
var may_states = Ict15j.Digitize(xsMay, 16);
Console.WriteLine($"May : {may_states.Length} pas, n_unique={may_states.Distinct().Count()}");

Gray-Scott : 4096 pixels, sum=90


Axelrod (repl baseline) : 400 pas, sum=398


Axelrod (evo regenere)  : 401 pas, coop[0,259,0,944], n_unique=8


Grokking : 400 pas, n_unique=4


May : 2000 pas, n_unique=16


### Le banc de quatre substrats : quatre régimes pour tester la cohérence

| Substrat | Régime | Alphabet | Observable |
|----------|--------|----------|------------|
| **Gray-Scott** | motif spatial émergent (Turing) | 2 | `V_final` binarisé (pattern / fond) |
| **Axelrod** (régénéré) | sélection-mutation (ICT-13 §6) | 8 | taux de coopération quantifié |
| **Axelrod** (baseline) | réplicateur end-of-cycle | 2 | stabilité du dominant (dégénéré) |
| **Grokking** | crossover haute-entropie → compression | 4 | marche biaisée |
| **May (ICT-8)** | pâturage bistable | 16 | biomasse quantifiée |

C'est le même banc que ICT-15c, rappelé pour rendre ce notebook
auto-contenu. La diversité (du binaire au 16-quantiles, du motif spatial à
la transition d'apprentissage) est volontaire : c'est ce banc qui permet
de tester si le discriminant Čech détecte une **différence structurelle
persistante** d'un régime à l'autre — ou si, comme la SVD, il se laisse
tromper par le rang spectral du contre-exemple axelrod.

In [3]:
// --- Calcul des sections locales (30 fenetres x 3 proxys) ---
var substrats_data = new Dictionary<string, (int[] states, int nSymbols)> {
    ["gray_scott"] = (gray_scott_states, 2),
    ["axelrod"] = (axelrod_states, 8),
    ["axelrod_repl"] = (axelrod_repl_states, 2),
    ["grokking"] = (grokking_states, 4),
    ["may"] = (may_states, 16)
};
var substrats_sections = new Dictionary<string, Dictionary<string, List<double>>>();
foreach (var entry in substrats_data) {
    var sections = Ict15j.WindowedProxySignature(entry.Value.states, entry.Value.nSymbols, 30);
    substrats_sections[entry.Key] = sections;
    var gaps = sections["spectral_gap"];
    Console.WriteLine($"{entry.Key,12}: 30 fenetres ; spectral_gap range=[{gaps.Min():F3}, {gaps.Max():F3}]");
}

  gray_scott: 30 fenetres ; spectral_gap range=[0,109, 0,508]


     axelrod: 30 fenetres ; spectral_gap range=[0,202, 0,532]


axelrod_repl: 30 fenetres ; spectral_gap range=[0,500, 1,100]


    grokking: 30 fenetres ; spectral_gap range=[0,246, 1,000]


         may: 30 fenetres ; spectral_gap range=[0,165, 0,500]


### Fenêtrage et z-score : préparer le nuage de points

On découpe chaque trajectoire en **30 fenêtres consécutives** (≈ 130 pixels
Gray-Scott, ≈ 13 pas Axelrod/Grokking, ≈ 66 pas May). Sur chaque fenêtre on
calcule les 3 proxys. Pour éviter qu'un proxy à grande amplitude (par
exemple `sensitivity_max` qui peut atteindre 4.0 sur alphabet 16) ne
domine les distances, on **z-score** chaque proxy avant de mesurer la
distance euclidienne entre fenêtres : `(x - mean) / std`.

La **normalisation par proxy** est cruciale : sans elle, le nuage de points
serait dominé par l'axe `sensitivity_max`, et la structure du nerf Čech ne
refléterait que les variations de ce seul proxy — la discrimination
s'évanouirait.

In [4]:
// --- Calcul du b1 sur chaque substrat (filtration Rips complete) ---
public class Simplexe15j
{
    public int A, B, C, Dimension;
    public double Filtration;
}
public class Resultat15j
{
    public int b1, b1_n_classes, n_edges, n_triangles, b0;
    public double b1_max_persistence, epsilon, b1_persistence_sum;
}
public static class Rips15j
{
    public static Resultat15j Calculer(Dictionary<string,List<double>> sections, double quantile = 0.55)
    {
        var colonnes = sections.Values.ToArray();
        int n = colonnes[0].Count, p = colonnes.Length;
        var points = new double[n,p];
        // Normalisation par proxy, avec le même seuil de variance que la source.
        for (int j = 0; j < p; j++) {
            double moyenne = colonnes[j].Average();
            double ecart = Math.Sqrt(colonnes[j].Select(x => (x-moyenne)*(x-moyenne)).Average());
            if (ecart <= 1e-12) ecart = 1.0;
            for (int i = 0; i < n; i++) points[i,j] = (colonnes[j][i]-moyenne)/ecart;
        }
        var distances = new double[n,n];
        var positives = new List<double>();
        for (int i = 0; i < n; i++) for (int j = i+1; j < n; j++) {
            double somme = 0;
            for (int k = 0; k < p; k++) somme += (points[i,k]-points[j,k])*(points[i,k]-points[j,k]);
            distances[i,j] = distances[j,i] = Math.Sqrt(somme);
            if (distances[i,j] > 0 && double.IsFinite(distances[i,j])) positives.Add(distances[i,j]);
        }
        double epsilon = positives.Count == 0 ? 0 : Ict15j.Quantile(positives.ToArray(), quantile);
        var simplices = new List<Simplexe15j>();
        for (int i = 0; i < n; i++) simplices.Add(new Simplexe15j { A=i, B=-1, C=-1, Dimension=0, Filtration=0 });
        for (int i = 0; i < n; i++) for (int j = i+1; j < n; j++) simplices.Add(new Simplexe15j { A=i, B=j, C=-1, Dimension=1, Filtration=distances[i,j] });
        for (int i = 0; i < n; i++) for (int j = i+1; j < n; j++) for (int k = j+1; k < n; k++) simplices.Add(new Simplexe15j { A=i, B=j, C=k, Dimension=2, Filtration=Math.Max(distances[i,j], Math.Max(distances[i,k], distances[j,k])) });
        simplices = simplices.OrderBy(s => s.Filtration).ThenBy(s => s.Dimension).ThenBy(s => s.A).ThenBy(s => s.B).ThenBy(s => s.C).ToList();
        var index = new Dictionary<(int,int,int),int>();
        for (int i = 0; i < simplices.Count; i++) { var s = simplices[i]; index[(s.A,s.B,s.C)] = i; }
        // Réduction des colonnes de bord sur F2 : pivot -> colonne réduite.
        var pivots = new Dictionary<int,HashSet<int>>();
        var naissance = new bool[simplices.Count];
        var intervalles = new List<(double debut,double fin)>();
        for (int col = 0; col < simplices.Count; col++) {
            var s = simplices[col]; var bord = new HashSet<int>();
            if (s.Dimension == 1) { bord.Add(index[(s.A,-1,-1)]); bord.Add(index[(s.B,-1,-1)]); }
            if (s.Dimension == 2) { bord.Add(index[(s.A,s.B,-1)]); bord.Add(index[(s.A,s.C,-1)]); bord.Add(index[(s.B,s.C,-1)]); }
            while (bord.Count > 0 && pivots.ContainsKey(bord.Max())) bord.SymmetricExceptWith(pivots[bord.Max()]);
            if (bord.Count == 0) naissance[col] = true;
            else {
                int pivot = bord.Max(); pivots[pivot] = bord;
                if (s.Dimension == 2 && naissance[pivot] && s.Filtration > simplices[pivot].Filtration)
                    intervalles.Add((simplices[pivot].Filtration,s.Filtration));
            }
        }
        var resultat = new Resultat15j { epsilon=epsilon, b1_n_classes=intervalles.Count };
        if (intervalles.Count > 0) {
            resultat.b1_max_persistence = intervalles.Max(x => x.fin-x.debut);
            resultat.b1_persistence_sum = intervalles.Sum(x => x.fin-x.debut);
        }
        if (epsilon <= 0) { resultat.b0 = n; return resultat; }
        resultat.n_edges = simplices.Count(s => s.Dimension == 1 && s.Filtration <= epsilon);
        resultat.n_triangles = simplices.Count(s => s.Dimension == 2 && s.Filtration <= epsilon);
        resultat.b1 = intervalles.Count(x => x.debut <= epsilon && epsilon < x.fin);
        // Les composantes sont calculées par union des arêtes présentes au seuil.
        var parent = Enumerable.Range(0,n).ToArray();
        Func<int,int> racine = null;
        racine = x => parent[x] == x ? x : (parent[x] = racine(parent[x]));
        foreach (var s in simplices.Where(s => s.Dimension == 1 && s.Filtration <= epsilon)) { int a = racine(s.A), b = racine(s.B); if (a != b) parent[a] = b; }
        resultat.b0 = Enumerable.Range(0,n).Select(racine).Distinct().Count();
        return resultat;
    }
}
var chrono15j = System.Diagnostics.Stopwatch.StartNew();
var results = new Dictionary<string,Resultat15j>();
foreach (var entree in substrats_sections) results[entree.Key] = Rips15j.Calculer(entree.Value);
chrono15j.Stop();
Console.WriteLine($"Temps de calcul : {chrono15j.Elapsed.TotalSeconds:F2}s\n");
Console.WriteLine($"{ "substrat",12} | {"b1",4} | {"b1_max_pers",12} | {"b1_n_cl",8} | {"n_edges",8} | {"n_tri",6}");
Console.WriteLine(new string('-',70));
foreach (var entree in results) {
    var r = entree.Value;
    Console.WriteLine($"{entree.Key,12} | {r.b1,4} | {r.b1_max_persistence,12:F4} | {r.b1_n_classes,8} | {r.n_edges,8} | {r.n_triangles,6}");
}

Temps de calcul : 0,27s



    substrat |   b1 |  b1_max_pers |  b1_n_cl |  n_edges |  n_tri


----------------------------------------------------------------------


  gray_scott |    0 |       0,0000 |        0 |      302 |   2099


     axelrod |    0 |       0,4147 |        5 |      239 |    971


axelrod_repl |    0 |       0,0000 |        0 |      406 |   3654


    grokking |    0 |       0,1766 |        4 |      245 |   1125


         may |    0 |       0,1136 |        3 |      241 |   1057


### Lecture du résultat : les deux observables Axelrod

La sortie précédente donne les mesures de la filtration Rips pour les cinq substrats. Les trajectoires stochastiques utilisent une graine fixe, mais le générateur C# diffère de celui de NumPy : les résultats numériques ne coïncident donc pas nécessairement avec ceux du jumeau Python.

- La baseline `axelrod_repl` ne présente aucune classe H¹ persistante. L'observable binaire de stabilité du dominant est presque constant après convergence du réplicateur.
- L'Axelrod régénéré présente la persistance maximale la plus élevée du banc. Son observable est le taux de coopération quantifié d'une population évolutive.
- `grokking` et `may` présentent également des classes persistantes au-dessus du seuil du verdict.

La comparaison porte sur la **forme de l'observable** : un résultat topologique trivial ne suffit pas à caractériser toute la dynamique du système sous-jacent.

In [5]:
// --- Verdict de discrimination (b1_max_persistence, tolerance=0.05) ---
var b1_by_substrat = results.ToDictionary(e => e.Key, e => e.Value.b1_max_persistence);
var valeursB1 = b1_by_substrat.Values.ToArray();
int n_nontrivial = valeursB1.Count(x => x > 0.05);
double mean_b1 = valeursB1.Average();
double std_b1 = Math.Sqrt(valeursB1.Select(x => (x-mean_b1)*(x-mean_b1)).Average());
double range_b1 = valeursB1.Max()-valeursB1.Min();
string verdict = n_nontrivial == 0 ? "TRIVIAL" : "NON_TRIVIAL";
Console.WriteLine("=== Verdict de discrimination ===");
Console.WriteLine("  metric_name: b1_max_persistence");
Console.WriteLine("  b1_by_substrat: " + string.Join(", ", b1_by_substrat.Select(e => $"{e.Key}={e.Value:R}")));
Console.WriteLine($"  n_nontrivial: {n_nontrivial}");
Console.WriteLine($"  mean_b1: {mean_b1:R}\n  std_b1: {std_b1:R}\n  range_b1: {range_b1:R}");
Console.WriteLine($"  b1_max: {valeursB1.Max():R}\n  b1_min: {valeursB1.Min():R}");
Console.WriteLine("  diverges_from_svd: None\n  rho_svd: None");
Console.WriteLine($"  verdict: {verdict}\n\nVERDICT FINAL = {verdict}");
Console.WriteLine($"  - n_nontrivial : {n_nontrivial} substrats avec b1_max_persistence > 0.05");
Console.WriteLine($"  - mean_b1 : {mean_b1:F4}, std_b1 : {std_b1:F4}");
Console.WriteLine($"  - range_b1 : {range_b1:F4}");

=== Verdict de discrimination ===


  metric_name: b1_max_persistence


  b1_by_substrat: gray_scott=0, axelrod=0,41469271303756816, axelrod_repl=0, grokking=0,17658229239852297, may=0,1135933817716358


  n_nontrivial: 3


  mean_b1: 0,14097367744154538
  std_b1: 0,15276578148298034
  range_b1: 0,41469271303756816


  b1_max: 0,41469271303756816
  b1_min: 0


  diverges_from_svd: None
  rho_svd: None


  verdict: NON_TRIVIAL

VERDICT FINAL = NON_TRIVIAL


  - n_nontrivial : 3 substrats avec b1_max_persistence > 0.05


  - mean_b1 : 0,1410, std_b1 : 0,1528


  - range_b1 : 0,4147


### Verdict falsifiable et critères préenregistrés

La mesure de persistance donne le verdict **NON_TRIVIAL** : trois substrats dépassent le seuil de 0,05, dont l'Axelrod régénéré. Sa persistance maximale est positive, mais son `b1` instantané au seuil ε vaut 0 : ces deux observables ne doivent pas être confondus.

Les critères se lisent séparément :

- **Relief sur l'Axelrod** : une classe H¹ persistante est observée pour l'Axelrod régénéré. En revanche, le tableau ne montre pas `b1 ≥ 1` à l'instantané ε pour ce substrat.
- **Divergence d'avec la SVD** : le critère préenregistré est une corrélation de Pearson inférieure à 0,9 entre les mesures de persistance et `s2_over_s1`. Aucune série `s2_over_s1` n'est fournie au verdict ici : `rho_svd` et `diverges_from_svd` valent `None`. Ce critère n'est donc pas évalué, et le verdict `NON_TRIVIAL` ne démontre pas une divergence d'avec la SVD.

À titre exploratoire, le rapport `range_b1 / std_b1` est calculable à partir des valeurs affichées. Ce calcul ne constitue pas ici une preuve de divergence d'avec la SVD.

La comparaison explicite avec la SVD est proposée à l'exercice 3. Le contraste entre les deux observables Axelrod illustre quant à lui la dépendance du discriminant à la représentation choisie.

In [6]:
// --- Visualisation : barplot de b1_max_persistence par substrat ---
#r "nuget: ScottPlot, 5.0.55"
using Microsoft.DotNet.Interactive.Formatting;
public class Graphique15j
{
    public string Svg;
    public Graphique15j(string svg) { Svg = svg; }
}
Formatter.Register<Graphique15j>((graphique, writer) => writer.Write(graphique.Svg), "image/svg+xml");
var plot15j = new ScottPlot.Plot();
var noms15j = results.Keys.ToArray();
var hauteurs15j = noms15j.Select(n => results[n].b1_max_persistence).ToArray();
var barres15j = plot15j.Add.Bars(hauteurs15j);
for (int i = 0; i < barres15j.Bars.Count; i++) barres15j.Bars[i].FillColor = hauteurs15j[i] == 0 ? ScottPlot.Colors.Red : ScottPlot.Colors.Green;
var seuil15j = plot15j.Add.HorizontalLine(0.05);
seuil15j.Color = ScottPlot.Colors.Orange;
plot15j.Axes.Bottom.TickGenerator = new ScottPlot.TickGenerators.NumericManual(Enumerable.Range(0,noms15j.Length).Select(i => (double)i).ToArray(), noms15j);
plot15j.Title($"Discriminant Čech — verdict = {verdict}, n_nontrivial = {n_nontrivial}");
plot15j.YLabel("b1_max_persistence (H1 Rips)");
plot15j.XLabel("Substrat");
var svg15j = plot15j.GetSvgXml(800,500);
display(new Graphique15j(svg15j));

Installing Packages ScottPlot

Svg


### Visualisation : barplot des b1_max_persistence

Les hauteurs sont calculées à partir des résultats de la filtration : l'Axelrod régénéré domine le banc, suivi de `grokking` puis de `may`. Gray-Scott et l'Axelrod baseline ont ici une persistance maximale nulle.

La ligne orange indique le seuil utilisé pour compter les substrats non triviaux. Ce choix de seuil est un paramètre du verdict ; l'exercice 2 propose d'en examiner la sensibilité.

La visualisation reprend les mesures du tableau : elle les rend plus faciles à comparer sans ajouter une mesure indépendante.

### Exercice 1 — Sensibilité au nombre de fenêtres

Le notebook agrège 30 fenêtres par substrat (par défaut). Le verdict
`NON_TRIVIAL` repose sur la persistance de classes H^1 dans trois substrats (**`axelrod` régénéré, `grokking`, `may`**). Mais
`30` est un choix : trop peu de fenêtres, le complexe Rips est trop petit
pour exposer des cycles ; trop de fenêtres, le bruit lissé noie les cycles
faibles (comme celui de `grokking`).

**Objectif.** Faire varier `n_windows ∈ {10, 30, 100}` et observer l'effet
sur le verdict. Le verdict reste-t-il `NON_TRIVIAL` dans les 3 cas ? Sinon,
à partir de quel `n_windows` bascule-t-il ?

**Indices.**
- `Ict15j.WindowedProxySignature(states, n_symbols, nWindows: N)` recalcule les
  sections ; puis `Rips15j.Calculer(sections)` applique le discriminant.
- Boucle sur `n_windows` et imprime `(n_windows, n_nontrivial, verdict)` pour
  chaque substrat.

In [7]:
// Exercice 1 — Sensibilite du verdict au nombre de fenetres.
// TODO etudiant : balayer n_windows dans [10, 30, 100] et imprimer
// (n_windows, verdict.verdict, verdict.n_nontrivial) pour chaque valeur.
object sweep_results = null;
Console.WriteLine("Exercice 1 a completer -- sweep n_windows attendu ici.");

Exercice 1 a completer -- sweep n_windows attendu ici.


### Exercice 2 — Tolérance du verdict (sensibilité au seuil 0.05)

La classification `n_nontrivial` repose sur `b1_max_persistence > 0.05`.
C'est un seuil **conservateur** choisi pour exclure les classes H^1 de
bruit numérique. Mais 0.05 est arbitraire.

**Objectif.** Balayer la tolérance dans `np.linspace(0.0, 0.15, 13)` et
observer à quel point la classification `n_nontrivial` change. Si
`n_nontrivial` reste à 1-2 sur toute la plage, le verdict est **robuste**.
S'il bascule de 1 à 3 sur un pas de 0.01, le verdict est **fragile**.

**Indices.**
- `discrimination_verdict(results, use_persistence=True)` ne prend pas la
  tolérance en argument, mais le critère `n_nontrivial` est calculable à la
  main : `sum(1 for v in verdict['b1_by_substrat'].values() if v > tol)`.
- Imprimez la table `(tol, n_nontrivial, verdict_manuel)` pour 13 valeurs.

In [8]:
// Exercice 2 — Tolerance du verdict.
// TODO etudiant : balayer la tolerance dans np.linspace(0.0, 0.15, 13)
// et imprimer le compte n_nontrivial pour chaque tolerance.
object tol_sweep = null;
Console.WriteLine("Exercice 2 a completer -- sweep tolerance attendu ici.");

Exercice 2 a completer -- sweep tolerance attendu ici.


### Exercice 3 — Comparaison avec le verdict SVD

Le verdict SVD (`ict.cech_obstruction.verdict`) déclare NON_TRIVIAL sur `axelrod` dans l'étude de référence. Dans cette exécution, le discriminant Čech donne une persistance maximale nulle à `axelrod_repl` et classe l'Axelrod régénéré comme non trivial.

**Objectif.** Charger le verdict SVD sur les substrats et calculer explicitement la divergence Čech vs SVD. Si SVD(`axelrod_repl`) est NON_TRIVIAL et Čech(`axelrod_repl`) est TRIVIAL, identifier cette divergence ; sinon, expliquer la comparaison obtenue.

**Indices.**
- Consulter ICT-15d pour la définition et les données nécessaires au verdict SVD.
- Calculez `(substrat, svd_verdict, cech_verdict)` pour les substrats comparables et comptez les divergences.

In [9]:
// Exercice 3 -- Comparaison verdict SVD vs verdict Cech.
// TODO etudiant : charger le verdict SVD (ict.cech_obstruction.verdict)
// et compter les divergences avec le verdict Cech.
object svd_vs_cech = null;
Console.WriteLine("Exercice 3 a completer -- comparaison SVD/Cech attendue ici.");

Exercice 3 a completer -- comparaison SVD/Cech attendue ici.


## Vérifiez votre compréhension

1. La baseline Axelrod présente une persistance maximale nulle tandis que l'Axelrod régénéré présente la plus grande persistance du banc. Que dit le nombre de triangles affiché pour la baseline, et pourquoi une persistance nulle ne falsifie-t-elle pas à elle seule la SVD ?
2. Le verdict global est **NON_TRIVIAL** : trois substrats sur cinq dépassent le seuil. Pourquoi `grokking` et `may` comptent-ils dans ce verdict, et comment leur classement dépend-il du seuil choisi ?
3. L'exercice 1 fait varier le nombre de fenêtres. Quelle conclusion serait menacée si la persistance de l'Axelrod régénéré s'effondrait avec davantage de fenêtres, et pourquoi ce test est-il utile ?

## À retenir

**Résultat synthétique** : dans cette exécution, l'Axelrod régénéré présente la persistance maximale la plus élevée du banc, suivi de `grokking` et de `may`. Ces trois substrats dépassent le seuil du verdict. Gray-Scott et la baseline Axelrod ont une persistance maximale nulle.

La baseline réplicateur fournit un exemple d'**observable dégénéré** : la stabilité du dominant devient presque constante. Une persistance nulle caractérise ici le nuage de proxys associé à cet observable ; elle ne suffit pas à falsifier le verdict SVD ni à caractériser toute la dynamique sous-jacente.

À l'inverse, l'observable de coopération de la population évolutive donne un relief topologique dans cette exécution. Le contraste illustre une limite méthodologique : un verdict topologique trivial peut refléter aussi bien le système étudié que le choix de son observable.

## Attestation de parité avec la référence Python

Les comparaisons ci-dessous portent sur les sorties de référence Python et sur les sorties C# de cette exécution. Les écarts sont donnés dans le sens **C# moins Python**. Les trajectoires sont stochastiques : la graine déclarée est identique, mais le générateur C# diffère de NumPy. Aucune tolérance numérique arbitraire ne permet donc d'attester l'identité des trajectoires. Pour les entiers et verdicts comparables, l'égalité est exacte ; pour les nombres affichés à quatre décimales, la comparaison tient compte de la précision d'affichage, sans prétendre absorber les différences de tirage.

| Section et observable | Python | C# | Écart ou comparaison |
|---|---:|---:|---|
| Substrats — pixels Gray-Scott actifs | 0 | 90 | +90 ; trajectoires différentes |
| Substrats — somme baseline Axelrod | 400 | 398 | −2 ; trajectoires différentes |
| Substrats — minimum / maximum de coopération | 0,261 / 0,938 | 0,259 / 0,944 | −0,002 / +0,006, valeurs arrondies |
| Sections — plage du gap Gray-Scott | 0,500–0,500 | 0,109–0,508 | −0,391 / +0,008 |
| Sections — plage du gap Axelrod | 0,182–0,548 | 0,202–0,532 | +0,020 / −0,016 |
| Sections — plage du gap baseline | 0,500–0,500 | 0,500–1,100 | 0,000 / +0,600 |
| Sections — plage du gap Grokking | 0,360–0,787 | 0,246–1,000 | −0,114 / +0,213 |
| Sections — plage du gap May | 0,168–0,500 | 0,165–0,500 | −0,003 / 0,000 |
| Rips — persistance maximale Gray-Scott | 0,0000 | 0,0000 | 0,0000 |
| Rips — persistance maximale Axelrod | 0,3989 | 0,4147 | +0,0158 |
| Rips — persistance maximale baseline | 0,0000 | 0,0000 | 0,0000 |
| Rips — persistance maximale Grokking | 0,1100 | 0,1766 | +0,0666 |
| Rips — persistance maximale May | 0,0985 | 0,1136 | +0,0151 |
| Verdict — nombre de substrats non triviaux | 3 | 3 | égalité exacte |
| Verdict — moyenne / écart-type | 0,1215 / 0,1464 | 0,1410 / 0,1528 | +0,0195 / +0,0064 |
| Verdict — amplitude | 0,3989 | 0,4147 | +0,0158 |
| Verdict final | NON_TRIVIAL | NON_TRIVIAL | égalité exacte |

Les écarts entre valeurs arrondies peuvent différer d'une unité au dernier chiffre de l'écart calculé avant arrondi. Les nombres de classes, d'arêtes et de triangles diffèrent aussi : notamment, pour Gray-Scott, Python affiche 0 classe, 435 arêtes et 4 060 triangles, contre 0, 302 et 2 099 en C#. Pour la baseline Axelrod, les triplets sont respectivement (0, 435, 4 060) et (0, 406, 3 654). Ces différences ne sont pas certifiées équivalentes par la seule concordance du verdict.

**Périmètre de l'attestation.** Les sorties confirment la concordance du verdict et du nombre de substrats non triviaux, mais pas une parité numérique complète. En particulier, les entrées stochastiques ne sont pas identiques ; dans l'évolution de population, la source appelle la stratégie GTFT sans lui passer le générateur du match, tandis que ce port lui passe le générateur de l'évolution. La stricte équivalence du protocole reste donc à établir. La comparaison avec la SVD n'est pas évaluée : aucune série `s2_over_s1` n'est fournie au verdict.

## Pour poursuivre l'analyse

La robustesse au nombre de fenêtres, la sensibilité au seuil de tolérance et la comparaison explicite avec le verdict SVD substrat par substrat restent à examiner dans les exercices.

## Discipline de nommage

L'objet n'est **pas nommé**. On utilise « discriminant Čech », terme technique, sans attribuer de nom phénoménologique à l'objet.

## Références

- Protocole : issue [#12257](https://github.com/jsboige/CoursIA/issues/12257) ; régénération de l'observable Axelrod : issue [#12673](https://github.com/jsboige/CoursIA/issues/12673) ; série ICT : issue [#4588](https://github.com/jsboige/CoursIA/issues/4588).
- Mesure antérieure comparée : `ict.cech_obstruction.verdict` ; substrat Axelrod évolutif décrit dans ICT-13 §6.

## Pour aller plus loin

- L'audit du générateur (contrôle négatif) qui vide le verdict de sa lecture « nouveauté » : [ICT-15l](ICT-15l-IndependanceGenerateur-Python.ipynb).
- La phase Čech dont ce nerf hérite : [ICT-15d](ICT-15d-CechObstruction-Python.ipynb) ; la phase zéro des proxys : [ICT-15c](ICT-15c-MetaProxyObstruction-Python.ipynb).
- Le substrat Axelrod évolutif est décrit dans ICT-13 §6 : [ICT-13](../ICT-13-AxelrodStrategicMorphodynamics-Python.ipynb).
- **Jumeau Python** → [ICT-15j-NerveDiscriminant-Python](ICT-15j-NerveDiscriminant-Python.ipynb) — le carnet d'origine, en Python.
- Revenir au [cadrage de la série](../ICT-0-Framing.md).